# Garmin data: insights and freshness

Checks how fresh your Garmin data is (when your watch last synced, which metrics have arrived for which days) and gives an overview of the data itself. Everything is read from the dbt marts in the local warehouse; nothing calls Garmin.

**Before running:**
1. Refresh the warehouse: `uv run ingest garmin && uv run transform`.
2. Select the project's `.venv` as the kernel.


In [ ]:
import logging
from datetime import date, timedelta

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from garminreader.dashboard import insights as ins
from garminreader.dashboard import queries

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

N_DAYS = 30
TODAY = date.today()
START = TODAY - timedelta(days=N_DAYS - 1)

## 1. Sync freshness

When Garmin last received data, as seen at the latest ingest, plus the ingest itself. Ages are relative to now. A signal is stale when it is older than its threshold, or when it has a threshold but no timestamp. The latest activity has no threshold, since rest days are normal. Thresholds live in `rpt_garmin_freshness`; `uv run transform source freshness` checks the raw table too.


In [ ]:
signals = queries.freshness()
signals.style.format({"age_hours": "{:.1f}"}).apply(
    lambda row: ["background-color: #f8d7da" if row["is_stale"] else "" for _ in row], axis=1
)

## 2. Load daily metrics and activities

From `fct_daily_health`, `fct_activities` and `rpt_current_status`, the same marts the dashboard uses.


In [ ]:
daily_df = queries.daily_health(START, TODAY)
activities_df = queries.activities(START, TODAY)
current_status = queries.current_status()

display(Markdown(f"Loaded **{len(daily_df)} days** and **{len(activities_df)} activities**."))

## 3. Completeness per metric

Which metrics are present for which days. Gaps on the most recent day usually mean the watch has not synced yet; gaps further back mean the watch was not worn or the feature was not available.

In [ ]:
coverage = queries.metric_coverage(START, TODAY)
grid = coverage.pivot(index="metric", columns="calendar_date", values="is_present")
fig = px.imshow(
    grid.astype(int),
    color_continuous_scale=[[0, "#e5e7eb"], [1, "#2563eb"]],
    aspect="auto",
    labels={"x": "date", "y": "metric", "color": "present"},
    title="Data present per metric and day",
)
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
queries.metric_lag()

## 4. Data overview


In [ ]:
KEY_METRICS = [
    "resting_hr",
    "hrv_last_night_avg",
    "sleep_score",
    "body_battery_highest",
    "avg_stress",
    "steps",
    "readiness_score",
]
daily_df[KEY_METRICS].describe().T.round(1)

In [ ]:
TREND_METRICS = ["resting_hr", "hrv_last_night_avg", "sleep_score", "body_battery_highest", "avg_stress"]

trend = daily_df.set_index("date")[TREND_METRICS]
rolling = trend.rolling(7, min_periods=3).mean().add_suffix(" (7d avg)")
long = pd.concat([trend, rolling], axis=1).reset_index().melt(id_vars="date", var_name="series", value_name="value")
long["metric"] = long["series"].str.replace(" (7d avg)", "", regex=False)
long["kind"] = long["series"].str.contains("7d avg").map({True: "7-day average", False: "daily"})

fig = px.line(
    long,
    x="date",
    y="value",
    color="kind",
    facet_row="metric",
    height=220 * len(TREND_METRICS),
    title="Recovery metrics with 7-day average",
)
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
if activities_df.empty:
    display(Markdown("No activities in this range."))
else:
    weekly = (
        activities_df.assign(week=activities_df["date"].dt.to_period("W").dt.start_time)
        .groupby(["week", "type"], as_index=False)["duration_min"]
        .sum()
    )
    px.bar(weekly, x="week", y="duration_min", color="type", title="Weekly training minutes by type").show()

## 5. Insights

The same plain-language insights the dashboard shows, computed on this range.

In [ ]:
for insight in ins.build_all_insights(daily_df, activities_df, current_status):
    display(Markdown(f"**{insight.title}** ({insight.direction})  \n{insight.finding}"))